# 04 · The hybrid: a neural network inside the ODE (UDE)

**Goal:** add a learned term to the physics, train it, and then **interrogate what it learned**.
That last part leads straight to the project's most important finding.

**Time:** ~1 h · **You'll write:** `ZeroInitMLP`, `grad_wrt_state`

**How the exercises work.** Each one is a function with `raise NotImplementedError`. Replace that
with your code, then run the cell below it. `use(...)` checks your function against an independent
reference (the project's own code, autograd, or a closed form):
- `[ok]` → your version is used from then on.
- `[todo]` / `[fail]` → the notebook falls back to the reference solution so you can keep going.

Hints are folded under each exercise. Solutions are in `tutorials/solutions.py`; try first.

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "tutorials":
    os.chdir("..")
sys.path.insert(0, "tutorials")

import numpy as np
import torch
import matplotlib.pyplot as plt
from tutor import use, close
import solutions as sol

torch.set_num_threads(4)
plt.rcParams.update({"figure.figsize": (10, 3.2), "axes.grid": True, "grid.alpha": 0.3})

from biotwin.data.dataset import Cohort, standard_sets, subset, LH, H
from biotwin.models.mechanistic import MinimalModel, CorrectionNet
from biotwin.train import fit, predict
cohort = Cohort.load("data/cohort.npz")
S = standard_sets(cohort, seed=0)
te = S["test_eval"]
rmse60 = lambda p: (p[:, -1] - te["y"][:, -1]).pow(2).mean().sqrt().item()

## 1. Idea

The 7-state model is wrong in known ways: the real simulator has 13 states, nonlinear gastric
emptying, saturating glucose uptake and renal clearance. Rather than replace physics with a black
box, add a small network to the glucose equation:

$$\dot G = \underbrace{-S_G (G - G_b) - X G + c_g k_d D_2}_{\text{physics}} \;+\; \underbrace{f_{NN}(\text{state}, \text{recent CGM slope})}_{\text{learned residual dynamics}}$$

This is a **universal differential equation** (Rackauckas et al., 2020). The network is trained
*through* the solver, so it learns a correction to the dynamics, not to the output.

One design choice matters a lot: **initialise the last layer to zero.** Then at step 0 the hybrid
*is* the physics model, and the network only grows where the data asks for it.

## Exercise 1 — `ZeroInitMLP(n_in, hidden)`

An MLP with layers `n_in → hidden → hidden → 1` and `tanh` activations, whose **last** linear
layer has zero weight and bias. `forward(x)` takes `[B, n_in]` and returns `[B]`.


<details><summary><b>Hint</b></summary>

`torch.nn.init.zeros_(layer.weight)` and `zeros_(layer.bias)`. Remember `.squeeze(-1)`.

</details>

In [ ]:
class ZeroInitMLP(torch.nn.Module):
    def __init__(self, n_in, hidden):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError

def check(cls):
    net = cls(8, 32)
    x = torch.randn(5, 8)
    out = net(x)
    assert out.shape == (5,), f"output shape {tuple(out.shape)}, expected (5,)"
    close(out, torch.zeros(5), what="output at init")
    assert sum(p.numel() for p in net.parameters()) == 1377, "expected 8*32+32 + 32*32+32 + 32+1 = 1377 parameters"
    (out - 1).pow(2).sum().backward()
    assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in net.parameters()), "no gradient flows"

ZeroInitMLP = use(ZeroInitMLP, sol.ZeroInitMLP, check)

Gradients still flow even though the output starts at exactly zero. The last layer's gradient
is (hidden activations) × (error), which isn't zero. The project's `CorrectionNet` is this MLP
plus input scaling: it takes the 7 states and the recent CGM slope, normalised to order 1.

## 2. Build and train the hybrid

Start from the fitted ODE, so physics is already as good as it gets, then train everything
jointly for one minute on a subset.

In [ ]:
ode = MinimalModel(); ode.load_state_dict(torch.load("results/checkpoints/ode.pt"))
hyb = MinimalModel(CorrectionNet(32))
hyb.log_theta.data.copy_(ode.log_theta.data)
w = subset(te, torch.arange(len(te["y"])) < 200)
with torch.no_grad():
    print("hybrid == ODE at initialisation:", torch.allclose(hyb(w), ode(w)))

small_tr = subset(S["train"], torch.arange(len(S["train"]["y"])) % 12 == 0)
small_va = subset(S["val"], torch.arange(len(S["val"]["y"])) % 4 == 0)
hyb, hist = fit(hyb, small_tr, small_va, epochs=10, lr=3e-3, bs=256, gen=torch.Generator().manual_seed(0),
                max_minutes=1.0, verbose=False)
print("val RMSE (all horizons) by epoch:", [round(h["val_rmse"], 1) for h in hist])

In [ ]:
full = MinimalModel(CorrectionNet(32)); full.load_state_dict(torch.load("results/checkpoints/hybrid.pt")); full.eval()
print(f"test RMSE 60 min: ODE {rmse60(predict(ode, te)):.1f} | hybrid, 1 min here {rmse60(predict(hyb, te)):.1f} | "
      f"hybrid, full 6-min run {rmse60(predict(full, te)):.1f}")

The hybrid improves the 60-minute RMSE by about 4 mg/dL with 1.4 k extra parameters. By that
metric it's a success. Now the more important question: **what did the network learn?**

## 3. Interrogate the network

The physics says insulin action $X$ lowers glucose: $\partial \dot G_{phys} / \partial X = -G < 0$.
The network can change that. Compute $\partial f_{NN}/\partial X$ at the states the model actually
visits.

## Exercise 2 — `grad_wrt_state(fn, s, ctx, i)`

Return $\partial\, \text{fn}(s, \text{ctx})_b / \partial s_{b,i}$ for every row $b$, shape `[B]`.
Each row's output depends only on that row's state.


<details><summary><b>Hint</b></summary>

Make a fresh leaf: `s = s.detach().clone().requires_grad_(True)`. Because rows are independent, the gradient of `out.sum()` gives every row's derivative in one backward pass: `torch.autograd.grad(out.sum(), s)[0][:, i]`.

</details>

In [ ]:
def grad_wrt_state(fn, s, ctx, i):
    # TODO
    raise NotImplementedError

def check(f):
    g = torch.Generator().manual_seed(0)
    s, ctx = torch.rand(16, 7, generator=g) * torch.tensor([1, 1, 1, .02, 10, 10, 200.]), torch.randn(16, generator=g)
    eps = 1e-3
    e = torch.zeros(7); e[3] = eps
    with torch.no_grad():
        fd = (full.correction(s + e, ctx) - full.correction(s - e, ctx)) / (2 * eps)
    close(f(full.correction, s, ctx, 3), fd, tol=1e-2, what="dNN/dX")

grad_wrt_state = use(grad_wrt_state, sol.grad_wrt_state, check)

In [ ]:
with torch.no_grad():
    _, states = full(te, return_states=True)          # [N, H, 7]
s0 = states[:, 0]                                      # state 5 min into each forecast
g = te["cgm_hist"]; ctx = (g[:, -1] - g[:, -4]) / 15
dnn_dX = grad_wrt_state(full.correction, s0, ctx, 3)
total = dnn_dX - s0[:, 6]                              # physics part: d(-X G)/dX = -G

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.3))
a1.hist(dnn_dX.numpy(), 60); a1.set_title("∂f_NN/∂X  (physics says: add nothing)")
a2.hist(total.numpy(), 60, color="tab:red"); a2.axvline(0, c="k"); a2.set_title("∂(dG/dt)/∂X = −G + ∂f_NN/∂X")
plt.tight_layout(); plt.show()
print(f"∂f_NN/∂X > 0 in {(dnn_dX > 0).float().mean():.0%} of states")
print(f"net effect of more insulin action is to RAISE glucose in {(total > 0).float().mean():.0%} of states")
for nm in S["split"]["test"]:
    msk = te["pid"] == cohort.idx(nm)
    print(f"  {nm:15s} {(total[msk] > 0).float().mean():.0%}")

**The network learned to push back against insulin.** Its derivative with respect to $X$ is
positive almost everywhere, and in a sizeable fraction of states it outweighs the physics, so the
model's glucose rises when insulin action rises.

How can that happen while RMSE improves? In the training data, insulin is **confounded** with
meals: boluses come with food. A term that says "high insulin action → glucose goes up" partly
encodes "a meal is being digested". It fits observed trajectories, and it's causally wrong.

(This is the instantaneous partial derivative. Insulin also acts through the other states, and
the network's dependence on $I_p$ points the other way. Notebook 07 tests the *end-to-end*
effect of a bolus directly, and finds violations too.)

Ways to stop it (the challenge in notebook 07):
- don't give the network the insulin states $I_{sc1}, I_{sc2}, I_p, X$ as inputs;
- force $\partial f_{NN}/\partial X \le 0$ (a monotone architecture, or a penalty);
- penalise the size of $f_{NN}$ relative to the physics terms;
- train on simulated interventions, which break the confounding.

## Quiz

Answer in your own words before opening the answers.

**1. Why initialise the last layer of the correction network to zero?**

<details><summary>Answer</summary>

So that training starts from the physics model (hybrid = ODE at step 0) and the network only adds what the data demands. It avoids a random initial term dominating the dynamics, and it gives a clean ablation: any difference from the ODE was learned.

</details>

**2. The network is trained through the ODE solver. What's the difference between correcting the dynamics (dG/dt) and correcting the output (G)?**

<details><summary>Answer</summary>

A dynamics correction is integrated over time, so it must be consistent across all horizons and interacts with the states. It also applies to new input sequences, which is what we need for what-if questions. An output correction is a static residual on top: easier to fit, but it says nothing about how interventions propagate.

</details>

**3. RMSE improved but ∂(dG/dt)/∂X > 0 in a fraction of states. Explain how both can be true.**

<details><summary>Answer</summary>

RMSE is measured on observed data, where insulin is confounded with meals. The network can use insulin-related states as a proxy for 'food being digested' and improve the fit, while the interventional effect of insulin (holding food fixed) gets the wrong sign. Accuracy on observational data doesn't validate causal structure.

</details>

**4. Why not drop the network's insulin inputs from the start?**

<details><summary>Answer</summary>

You can, and it's one of the fixes. The trade-off: some real model errors do depend on insulin (for example, the real simulator's insulin acts on the liver and on tissue with different delays). Removing those inputs may cost accuracy. You want the constraint that keeps the causal sign at the lowest accuracy cost, and that's an empirical question.

</details>

**5. Name one way to test a hybrid's causal behaviour without any real interventional data.**

<details><summary>Answer</summary>

Metamorphic tests: properties that must hold whatever the true values, such as 'more insulin, same everything else → lower glucose'. Apply them across many patients and starting states. Or compare with the physics-only model's intervention effects: their signs should agree.

</details>